In [1]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd
import time
import re
from PyPDF2 import PdfReader
from datetime import datetime
import tempfile
from pdfminer.pdfparser import PDFSyntaxError
import pdfplumber
from collections import defaultdict
import logging
import shutil

# Configure logging and start timer
start = time.time()
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
pd.set_option('display.max_rows', 500, 'display.max_columns', 500, 'display.max_colwidth', 1000)

# Setup directories
BASE_DIR = "nlb_financial_data"
ORIGINAL_DIR = os.path.join(BASE_DIR, "original")
CURRENT_DIR = os.path.join(BASE_DIR, "current")
OUTPUT_DIR = os.path.join(BASE_DIR, "output")

# Create subdirectories for categorized files
for subfolder in ['balance-sheet', 'income-statement']:
    os.makedirs(os.path.join(CURRENT_DIR, subfolder), exist_ok=True)
    print(f"Created directory: {os.path.join(CURRENT_DIR, subfolder)}")

for dir_path in [BASE_DIR, ORIGINAL_DIR, OUTPUT_DIR]:
    os.makedirs(dir_path, exist_ok=True)
    print(f"Created directory: {dir_path}")

# Log file for tracking processed files
PROCESSED_FILES_LOG = os.path.join(BASE_DIR, "processed_files.csv")

bank_urls = [
    'https://nlb-kos.com/sq/pasqyrat-financiare?page=1',
    'https://nlb-kos.com/sq/pasqyrat-financiare?page=2',
    'https://nlb-kos.com/sq/pasqyrat-financiare?page=3',
    'https://nlb-kos.com/sq/pasqyrat-financiare?page=4'
]
print(f"Target URLs: {len(bank_urls)} pages")

def load_processed_files():
    """Load list of already processed files from CSV"""
    if os.path.exists(PROCESSED_FILES_LOG):
        df = pd.read_csv(PROCESSED_FILES_LOG)
        return set(df['file_name'].tolist())
    return set()

def save_processed_file(file_name, status=1, reporting_date=None, current_name=None, is_corrupt=0):
    """Save processed file to log"""
    processed = load_processed_files()
    if file_name not in processed:
        df = pd.DataFrame({
            'file_name': [file_name],
            'processed_date': [datetime.now().strftime('%Y-%m-%d %H:%M:%S')],
            'status': [status],
            'reporting_date': [reporting_date],
            'current_name': [current_name],
            'is_corrupt': [is_corrupt]
        })
        if os.path.exists(PROCESSED_FILES_LOG):
            existing = pd.read_csv(PROCESSED_FILES_LOG)
            df = pd.concat([existing, df], ignore_index=True)
        df.to_csv(PROCESSED_FILES_LOG, index=False)

def scrape_pdf_links(url):
    """Scrape PDF links from NLB website (multi-page)"""
    print(f" Scraping: {url}")
    try:
        response = requests.get(url)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        pdf_links = []
        
        # First, find all links to report pages
        for link in soup.find_all('a', href=True):
            href = link['href']
            if 'pasqyra' in href.lower() and href.lower().startswith('https://nlb-kos.com/sq/pasqyrat/'):
                print(f"  Found report page: {href}")
                # Visit each report page to find actual PDF
                try:
                    report_response = requests.get(href)
                    report_soup = BeautifulSoup(report_response.text, 'html.parser')
                    for pdf_link in report_soup.find_all('a', href=True):
                        pdf_href = pdf_link['href']
                        if pdf_href.lower().endswith('.pdf') and pdf_href.lower().startswith('https://nlb-kos.com/storage/app/uploads/public'):
                            pdf_links.append(pdf_href)
                            print(f" Found PDF: {os.path.basename(pdf_href)}")
                except Exception as e:
                    print(f" Error visiting report page: {e}")
        
        return pdf_links
    except Exception as e:
        print(f"Error scraping {url}: {e}")
        return []

def download_pdf(url, local_dir):
    """Download PDF to local directory"""
    file_name = os.path.basename(url)
    local_path = os.path.join(local_dir, file_name)
    
    print(f" Downloading: {file_name}")
    try:
        response = requests.get(url)
        response.raise_for_status()
        
        with open(local_path, 'wb') as f:
            f.write(response.content)
        
        file_size = os.path.getsize(local_path) / 1024
        print(f"  Downloaded: {file_name} ({file_size:.1f} KB)")
        return local_path
    except Exception as e:
        print(f" Failed to download {file_name}: {e}")
        return None

def extract_date(text, filename):
    """Extract a date string from text or filename"""
    # Try to find date pattern like 31.12.2023 with possible single-digit days/months
    match = re.search(r'\d{1,2}[.]\d{1,2}[.]\d{4}', text)
    if match:
        date_parts = match.group().split('.')
        date_parts = [part.zfill(2) for part in date_parts]
        return '.'.join(date_parts)
    
    # Try standard date pattern
    match = re.search(r'\d{2}[./-]\d{2}[./-]\d{4}', text)
    if match:
        return match.group().replace('-', '.').replace('/', '.')
    
    # Try filename
    match = re.search(r'\d{2}[./-]\d{2}[./-]\d{4}', filename)
    if match:
        return match.group().replace('-', '.').replace('/', '.')
    
    return None

def download_all(force=False):
    """Download PDFs and categorize them locally"""
    processed = set() if force else load_processed_files()
    if force:
        print(" Force mode enabled: all files will be downloaded")
    
    # Collect all PDF links from all pages
    all_pdf_links = []
    for bank_url in bank_urls:
        pdf_links = scrape_pdf_links(bank_url)
        all_pdf_links.extend(pdf_links)
    
    if not all_pdf_links:
        print(" No PDF links found")
        return []
    
    new_files = []
    file_metadata = []

    print(f"\n Starting download of {len(all_pdf_links)} PDFs...")
    
    for i, pdf_url in enumerate(all_pdf_links, 1):
        file_name = os.path.basename(pdf_url)
        print(f"\n[{i}/{len(all_pdf_links)}] Processing: {file_name}")
        
        # Skip if already processed
        if file_name in processed and not force:
            print(f"  File already processed. Skipping...")
            continue

        new_files.append(file_name)
        
        # Download to original directory
        file_path = download_pdf(pdf_url, ORIGINAL_DIR)
        if not file_path:
            continue

        # Try to read metadata creation date
        dt = None
        try:
            with open(file_path, 'rb') as f:
                pdf_reader = PdfReader(f)
                metadata = pdf_reader.metadata
                if metadata:
                    for k, v in metadata.items():
                        if "/CreationDate" in k:
                            date_str = re.search(r'D:(\d+)', str(v))
                            if date_str:
                                dt = datetime.strptime(date_str.group(1), '%Y%m%d%H%M%S')
                                print(f"  PDF creation date: {dt.date()}")
                                break
        except Exception as e:
            print(f" Could not read metadata: {e}")

        # Extract date from PDF content
        date = None
        try:
            with pdfplumber.open(file_path) as pdf:
                full_text = ""
                for page in pdf.pages:
                    text = page.extract_text()
                    if text:
                        full_text += text + "\n"
                date = extract_date(full_text, file_name)
                
            if date:
                print(f"  Extracted reporting date: {date}")
            else:
                print(f"  Could not extract date")
                date = datetime.now().strftime('%d.%m.%Y')
        except Exception as e:
            print(f"  Error extracting text: {e}")
            date = datetime.now().strftime('%d.%m.%Y')

        # For NLB, we create both BS and IS from the same PDF
        bs_new_name = f'nlb_bs_{date}.pdf'
        is_new_name = f'nlb_is_{date}.pdf'
        
        # Copy to both categorized folders
        shutil.copy2(file_path, os.path.join(CURRENT_DIR, 'balance-sheet', bs_new_name))
        shutil.copy2(file_path, os.path.join(CURRENT_DIR, 'income-statement', is_new_name))
        
        file_metadata.append({
            'file_name': file_name,
            'file_path': file_path,
            'bs_current_name': bs_new_name,
            'is_current_name': is_new_name,
            'reporting_date': date,
            'creation_date': dt,
            'download_date': datetime.now(),
            'is_corrupt': 0
        })
        
        print(f"Created: {bs_new_name} and {is_new_name}")
        
        # Mark as processed
        save_processed_file(file_name, reporting_date=date, current_name=f"{bs_new_name}|{is_new_name}")

    # Save metadata
    if file_metadata:
        metadata_df = pd.DataFrame(file_metadata)
        metadata_path = os.path.join(OUTPUT_DIR, f"file_metadata_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv")
        metadata_df.to_csv(metadata_path, index=False)
        print(f"\n File metadata saved to {metadata_path}")

    if new_files:
        print(f"\n Downloaded {len(new_files)} new files.")
    else:
        print("\n No new files detected.")

    # Return list of all new current files
    current_files = []
    for m in file_metadata:
        current_files.append(m['bs_current_name'])
        current_files.append(m['is_current_name'])
    return current_files

def replace_negatives(value):
    """Convert parenthesized numbers to negative"""
    if pd.isna(value) or str(value).strip() in ['', '-', '0']:
        return '0'
    val = str(value).strip()
    if '(' in val and ')' in val:
        num = val.replace('(', '').replace(')', '').replace(',', '').strip()
        return f"-{num}"
    return val

# Define categories
income_statement_categories = [
    'Të hyrat nga interesi', 'Shpenzimet e interesit', 'Neto të hyrat nga interesi',
    'Të hyrat nga tarifat dhe komisionet', 'Shpenzimet e tarifave dhe komisioneve',
    'Neto të hyrat nga tarifat dhe komisionet', 'Neto të hyrat nga tregtimi me valuta të huaja',
    'Neto të hyrat nga tregtimi', 'Neto të hyrat nga instrumentet tjera financiare',
    'Neto të hyrat (shpenzimet) tjera operative', 'Gjithsej të hyrat',
    'Provizionet për humbjet nga kreditë','Provizione të tjera', 'Provizionet të tjera',
    'Fitimi (humbja) para tatimit', 'Fitimi/(humbja) para tatimit', 'Fitimi para tatimit',
    'Shpenzimet e tatimit në fitim', 'Fitimi (humbja) neto', 'Fitimi/(humbja) neto',
    'Fitimi neto', 'Të ardhurat tjera gjithëpërfshirëse', 'Gjithsej të ardhurat gjithëpërfshirëse',
    'Gjithsej te ardhurat gjithëpërfshirëse', 'Gjithsejt te ardhurat gjitheperfshirese per vitin',
    'Gjithsejt të ardhurat gjithëpërfshirëse për vitin',
    'Gjithsejt te ardhurat gjithëpërfshirëse per vitin', 'Fitimi neto pas tatimit',
    'Fitimi/(humbja) neto pas tatimit'
]

balance_sheet_categories = [
    "Paraja e gatshme dhe gjendja me BQK-në", "Kërkesat ndaj bankave", "Bonot e thesarit",
    "Mjetet jo qarkulluese të mbajtura për shitje", "Investimet në letra me vlerë",
    "Kreditë dhe paradhëniet ndaj klientëve", "Patundshmëritë dhe pajisjet",
    "Pasuritë e paprekshme", "Pasuritë tatimore të shtyra", "Pasuritë tjera",
    "Gjithsej pasuritë", "Depozitat e klientëve", "Detyrimet ndaj bankave",
    "Borgjet afatgjata", "Borxhet afataxhata", "Borxhet afatgjata",
    "Borxhet afatshkurtera", "Borxhet afatshkurtëra", "Borgjet afatshkurtera",
    "Borgjet afat shkurta", "Borgjet afatshkurta", "Borxhet afatshkurtëra",
    "Borxhi i ndërvarur", "Fondet tjera të huazuara",
    "Detyrimet tatimore të shtyra", "Detyrimet tjera", "Gjithsej detyrimet",
    "Kapitali aksionar", "Fitimi i mbajtur nga vitet paraprake",
    "Fitimi I mbajtur/(humbja) nga vitet paraprake", "Humbja nga vitet paraprake",
    "Rezervat e kapitalit", "Fitimi i mbajtur/(humbja) nga vitet paraprake",
    "Rezerva per vleren e tregut", "Fitimi/(humbja) e vitit aktual", "Fitimi i vitit aktual",
    "Përbërësit tjerë të ekuititetit", "Përbërësit tjerë të ekuitetit",
    "Gjithsej ekuiteti i aksionarëve", "Gjithsej ekuititeti I aksionarëve",
    "Gjithsej detyrimet dhe ekuiteti i aksionarëve", "Gjithsej detyrimet dhe ekuititeti i aksionarëve"
]

target_categories = {
    "Income Statement": income_statement_categories,
    "Balance Sheet": balance_sheet_categories,
}

# Data storage
extracted_data = defaultdict(list)

def process_pdf(pdf_path, filename):
    """Process PDF and extract financial data"""
    file_current_name = os.path.basename(filename)
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                text = page.extract_text()
                if not text:
                    continue
                    
                dt_report = extract_date(text, pdf_path)
                
                if dt_report:
                    # Convert to consistent format
                    try:
                        date_obj = datetime.strptime(dt_report, '%d.%m.%Y')
                        dt_report_formatted = date_obj.strftime('%Y-%m-%d')
                    except:
                        dt_report_formatted = dt_report
                else:
                    dt_report_formatted = datetime.now().strftime('%Y-%m-%d')
                
                for doc_type, categories in target_categories.items():
                    categories_processed = set()
                    
                    for line in text.split('\n'):
                        for category in categories:
                            if category.lower() in line.lower() and category not in categories_processed:
                                categories_processed.add(category)
                                
                                start_index = line.find(category) + len(category)
                                line_rest = line[start_index:]
                                values = re.findall(r'\(?\d+[,.]?\d*\)?', line_rest)
                                
                                if values and len(values) >= 2:
                                    extracted_data[doc_type].append({
                                        'Category': category,
                                        'PREVIOUS_QUARTER': values[0],
                                        'CURRENT_QUARTER': values[1],
                                        'DT_REPORT': dt_report_formatted,
                                        'FILE_NAME': filename
                                    })
                                    print(f"   Found {doc_type}: {category}")
                                    break
        print(f"    Processed: {filename}")
    except Exception as e:
        print(f"    Error processing {filename}: {e}")

def clean_numeric_values(df, prev_col, curr_col):
    """Clean and convert numeric columns"""
    for col in [prev_col, curr_col]:
        df[col] = df[col].astype(str).str.replace('-', '0', regex=False)
        df[col] = df[col].apply(replace_negatives)
        df[col] = df[col].str.replace(',', '')
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    return df

def main(force=False, extract_only=False):
    """Main execution function that returns unified DataFrames"""
    
    print("\n" + "="*60)
    print(" NLB BANK FINANCIAL DATA EXTRACTION")
    print("="*60)
    
    # Clear previous data
    global extracted_data
    extracted_data.clear()
    
    # Decide which files to process
    if extract_only:
        print("\n EXTRACT-ONLY MODE: Processing existing files")
        new_files = []
        for category in ['balance-sheet', 'income-statement']:
            category_dir = os.path.join(CURRENT_DIR, category)
            if os.path.exists(category_dir):
                category_files = [f for f in os.listdir(category_dir) if f.startswith('nlb_') and f.endswith('.pdf')]
                new_files.extend(category_files)
        print(f"Found {len(new_files)} existing files")
    else:
        print("\n DOWNLOAD MODE: Downloading and processing new files")
        new_files = download_all(force=force)
    
    if new_files or extract_only:
        # Process all files from both folders
        for category in ['balance-sheet', 'income-statement']:
            category_dir = os.path.join(CURRENT_DIR, category)
            if os.path.exists(category_dir):
                print(f"\n Processing {category} files...")
                for filename in os.listdir(category_dir):
                    if filename.startswith('nlb_') and filename.endswith('.pdf'):
                        if filename in new_files or extract_only:
                            file_path = os.path.join(category_dir, filename)
                            print(f"\n   🔍 Processing: {filename}")
                            process_pdf(file_path, filename)
        
        # Create DataFrames
        print("\n Creating DataFrames...")
        
        # Balance Sheet DataFrame
        full_bs = pd.DataFrame()
        if 'Balance Sheet' in extracted_data:
            full_bs = pd.DataFrame(extracted_data['Balance Sheet'])
        
        # Income Statement DataFrame
        full_is = pd.DataFrame()
        if 'Income Statement' in extracted_data:
            full_is = pd.DataFrame(extracted_data['Income Statement'])
        
        # Process Balance Sheet
        if not full_bs.empty:
            full_bs.columns = ['BALANCE_CATEGORY', 'PREVIOUS_BALANCE_VALUE', 'CURRENT_BALANCE_VALUE', 'DT_REPORT', 'FILE_NAME']
            full_bs = clean_numeric_values(full_bs, 'PREVIOUS_BALANCE_VALUE', 'CURRENT_BALANCE_VALUE')
            
            # Category mapping for balance sheet
            bs_map = {
                "Paraja e gatshme dhe gjendja me BQK-në": "20",
                "Kërkesat ndaj bankave": "21", 
                "Bonot e thesarit": "22",
                "Mjetet jo qarkulluese të mbajtura për shitje": "24",
                "Investimet në letra me vlerë": "23",
                "Kreditë dhe paradhëniet ndaj klientëve": "26",
                "Patundshmëritë dhe pajisjet": "27",
                "Pasuritë e paprekshme": "28",
                "Pasuritë tatimore të shtyra": "29",
                "Pasuritë tjera": "30",
                "Gjithsej pasuritë": "31",
                "Depozitat e klientëve": "32",
                "Detyrimet ndaj bankave": "33",
                "Borgjet afatgjata": "33",
                "Borxhet afataxhata": "33",
                "Borxhet afatshkurtera": "33",
                "Borxhet afatshkurtëra": "33",
                "Borgjet afatshkurtera": "33",
                "Borgjet afat shkurta": "33",
                "Borgjet afatshkurta": "33",
                "Borxhi i ndërvarur": "33",
                "Fondet tjera të huazuara": "34",
                "Detyrimet tatimore të shtyra": "35",
                "Detyrimet tjera": "36",
                "Gjithsej detyrimet": "37",
                "Kapitali aksionar": "38",
                "Fitimi i mbajtur nga vitet paraprake": "41",
                "Humbja nga vitet paraprake": "41",
                "Rezervat e kapitalit": "40",
                "Fitimi i mbajtur/(humbja) nga vitet paraprake": "41",
                "Fitimi I mbajtur/(humbja) nga vitet paraprake": "41",
                "Rezerva per vleren e tregut": "",
                "Fitimi/(humbja) e vitit aktual": "42",
                "Fitimi i vitit aktual": "42",
                "Përbërësit tjerë të ekuititetit": "43",
                "Përbërësit tjerë të ekuitetit": "43",
                "Gjithsej ekuiteti i aksionarëve": "44",
                "Gjithsej ekuititeti I aksionarëve": "44",
                "Gjithsej detyrimet dhe ekuiteti i aksionarëve": "45",
                "Gjithsej detyrimet dhe ekuititeti i aksionarëve": "45"
            }
            
            full_bs['CATEGORY_CODE'] = full_bs['BALANCE_CATEGORY'].map(bs_map)
            full_bs['BANK_ID'] = 7  # NLB Bank ID
            full_bs['STATEMENT_TYPE'] = 'BALANCE_SHEET'
            full_bs['DT_REPORT'] = pd.to_datetime(full_bs['DT_REPORT'], errors='coerce')
        
        # Process Income Statement
        if not full_is.empty:
            full_is.columns = ['INCOME_CATEGORY', 'PREVIOUS_INCOME_VALUE', 'CURRENT_INCOME_VALUE', 'DT_REPORT', 'FILE_NAME']
            full_is = clean_numeric_values(full_is, 'PREVIOUS_INCOME_VALUE', 'CURRENT_INCOME_VALUE')
            
            # Category mapping for income statement
            is_map = {
                'Të hyrat nga interesi': '1',
                'Shpenzimet e interesit': '2',
                'Neto të hyrat nga interesi': '3',
                'Të hyrat nga tarifat dhe komisionet': '4',
                'Shpenzimet e tarifave dhe komisioneve': '5',
                'Neto të hyrat nga tarifat dhe komisionet': '6',
                'Neto të hyrat nga tregtimi me valuta të huaja': '7',
                'Neto të hyrat nga tregtimi': '7',
                'Neto të hyrat nga instrumentet tjera financiare': '8',
                'Neto të hyrat (shpenzimet) tjera operative': '9',
                'Gjithsej të hyrat': '10',
                'Provizionet për humbjet nga kreditë': '11',
                'Provizionet të tjera': '13',
                'Provizione të tjera': '13',
                'Fitimi (humbja) para tatimit': '14',
                'Fitimi/(humbja) para tatimit': '14',
                'Fitimi para tatimit': '14',
                'Shpenzimet e tatimit në fitim': '15',
                'Fitimi (humbja) neto': '16',
                'Fitimi/(humbja) neto': '16',
                'Fitimi neto': '16',
                'Të ardhurat tjera gjithëpërfshirëse': '17',
                'Gjithsej të ardhurat gjithëpërfshirëse': '19',
                'Gjithsej te ardhurat gjithëpërfshirëse': '19',
                'Gjithsejt te ardhurat gjitheperfshirese per vitin': '19',
                'Gjithsejt të ardhurat gjithëpërfshirëse për vitin': '19',
                'Gjithsejt te ardhurat gjithëpërfshirëse per vitin': '19',
                'Fitimi neto pas tatimit': '64',
                'Fitimi/(humbja) neto pas tatimit': '64'
            }
            
            full_is['CATEGORY_CODE'] = full_is['INCOME_CATEGORY'].map(is_map)
            full_is['BANK_ID'] = 7  # NLB Bank ID
            full_is['STATEMENT_TYPE'] = 'INCOME_STATEMENT'
            full_is['DT_REPORT'] = pd.to_datetime(full_is['DT_REPORT'], errors='coerce')
        
        # Create unified DataFrame
        unified_dfs = []
        
        if not full_is.empty:
            is_unified = full_is.rename(columns={
                'INCOME_CATEGORY': 'ORIGINAL_CATEGORY',
                'PREVIOUS_INCOME_VALUE': 'PREVIOUS_VALUE',
                'CURRENT_INCOME_VALUE': 'CURRENT_VALUE'
            })
            is_unified['CATEGORY_TYPE'] = 'INCOME'
            unified_dfs.append(is_unified)
        
        if not full_bs.empty:
            bs_unified = full_bs.rename(columns={
                'BALANCE_CATEGORY': 'ORIGINAL_CATEGORY',
                'PREVIOUS_BALANCE_VALUE': 'PREVIOUS_VALUE',
                'CURRENT_BALANCE_VALUE': 'CURRENT_VALUE'
            })
            bs_unified['CATEGORY_TYPE'] = 'BALANCE'
            unified_dfs.append(bs_unified)
        
        if unified_dfs:
            final_df = pd.concat(unified_dfs, ignore_index=True)
            final_df['CURR_ID'] = 1
            final_df['DATA_SOURCE'] = 'NLB Bank'
            final_df['EXTRACTION_DATE'] = datetime.now().strftime('%Y-%m-%d')
            final_df['REPORT_DATE'] = final_df['DT_REPORT'].dt.strftime('%Y-%m-%d')
            
            # Drop rows with invalid dates or categories
            final_df = final_df.dropna(subset=['DT_REPORT', 'CATEGORY_CODE'])
            
            # Sort
            final_df = final_df.sort_values(['DT_REPORT', 'STATEMENT_TYPE', 'CATEGORY_CODE'])
            
            # Reorder columns
            column_order = ['BANK_ID', 'REPORT_DATE', 'DT_REPORT', 'STATEMENT_TYPE', 'CATEGORY_TYPE',
                          'CATEGORY_CODE', 'ORIGINAL_CATEGORY', 'PREVIOUS_VALUE', 'CURRENT_VALUE',
                          'CURR_ID', 'FILE_NAME', 'DATA_SOURCE', 'EXTRACTION_DATE']
            
            final_columns = [col for col in column_order if col in final_df.columns]
            final_df = final_df[final_columns]
            
            # Save outputs
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            
            # Save unified DataFrame
            unified_path = os.path.join(OUTPUT_DIR, f"nlb_unified_financial_data_{timestamp}.csv")
            final_df.to_csv(unified_path, index=False)
            print(f"\n Unified data saved to: {unified_path}")
            
            # Save individual files
            if not full_is.empty:
                is_path = os.path.join(OUTPUT_DIR, f"nlb_income_statement_{timestamp}.csv")
                full_is.to_csv(is_path, index=False)
            
            if not full_bs.empty:
                bs_path = os.path.join(OUTPUT_DIR, f"nlb_balance_sheet_{timestamp}.csv")
                full_bs.to_csv(bs_path, index=False)
            
            # Save Excel with multiple sheets
            excel_path = os.path.join(OUTPUT_DIR, f"nlb_financial_report_{timestamp}.xlsx")
            with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
                final_df.to_excel(writer, sheet_name='Unified_Data', index=False)
                if not full_is.empty:
                    full_is.to_excel(writer, sheet_name='Income_Statement', index=False)
                if not full_bs.empty:
                    full_bs.to_excel(writer, sheet_name='Balance_Sheet', index=False)
            
            print(f" Excel report saved to: {excel_path}")
            
            print(f"\n UNIFIED DATAFRAME CREATED:")
            print(f"   - Total rows: {len(final_df)}")
            print(f"   - Income Statement rows: {len(final_df[final_df['STATEMENT_TYPE'] == 'INCOME_STATEMENT'])}")
            print(f"   - Balance Sheet rows: {len(final_df[final_df['STATEMENT_TYPE'] == 'BALANCE_SHEET'])}")
            print(f"   - Unique report dates: {final_df['REPORT_DATE'].nunique()}")
            print(f"   - Date range: {final_df['REPORT_DATE'].min()} to {final_df['REPORT_DATE'].max()}")
            
            elapsed_time = time.time() - start
            print(f"\n Processing completed in {elapsed_time:.2f} seconds")
            
            return final_df, full_is, full_bs
        
        return pd.DataFrame(), full_is, full_bs
    
    return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

# Run the extraction
print(" Starting NLB Bank financial data extraction...")
final_df, income_df, balance_df = main(force=True, extract_only=False)

if not final_df.empty:
    print("\n" + "="*60)
    print(" FINAL UNIFIED DATAFRAME")
    print("="*60)
    print(f"Shape: {final_df.shape}")
    print("\nFirst 10 rows:")
    print(final_df.head(10))
    
    print("\n Summary by Statement Type:")
    print(final_df['STATEMENT_TYPE'].value_counts())
    
    print("\n Reports by Date:")
    print(final_df['REPORT_DATE'].value_counts().sort_index())
    
    print("\n Data Types:")
    print(final_df.dtypes)
else:
    print(" No data in final DataFrame")

c:\Users\Besarta Kurtaj\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.1) doesn't match a supported version!
  warnings.warn(


Created directory: nlb_financial_data\current\balance-sheet
Created directory: nlb_financial_data\current\income-statement
Created directory: nlb_financial_data
Created directory: nlb_financial_data\original
Created directory: nlb_financial_data\output
Target URLs: 4 pages
 Starting NLB Bank financial data extraction...

 NLB BANK FINANCIAL DATA EXTRACTION

 DOWNLOAD MODE: Downloading and processing new files
 Force mode enabled: all files will be downloaded
 Scraping: https://nlb-kos.com/sq/pasqyrat-financiare?page=1
 Scraping: https://nlb-kos.com/sq/pasqyrat-financiare?page=2
 Scraping: https://nlb-kos.com/sq/pasqyrat-financiare?page=3
 Scraping: https://nlb-kos.com/sq/pasqyrat-financiare?page=4
 No PDF links found
 No data in final DataFrame


In [ ]:
import os
import pandas as pd
import re
import time
import pdfplumber
from datetime import datetime
from collections import defaultdict

# ---------------------------
# CONFIGURATION
# ---------------------------

start = time.time()

# Update this to your actual path
BASE_DIR = "/Users/test/Desktop/Stuff/nlb_financial_data"
CURRENT_DIR = os.path.join(BASE_DIR, "current")
OUTPUT_DIR = os.path.join(BASE_DIR, "output")

for subfolder in ['balance-sheet', 'income-statement']:
    os.makedirs(os.path.join(CURRENT_DIR, subfolder), exist_ok=True)
    print(f" Directory ready: {os.path.join(CURRENT_DIR, subfolder)}")

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f" Output directory ready: {OUTPUT_DIR}")

# ---------------------------
# DATA STORAGE
# ---------------------------

extracted_data = defaultdict(list)

# ---------------------------
# FUNCTIONS
# ---------------------------

def extract_date(text, filename):
    """Extract a date string from text or filename"""
    match = re.search(r'\d{1,2}[.]\d{1,2}[.]\d{4}', text)
    if match:
        parts = [p.zfill(2) for p in match.group().split('.')]
        return '.'.join(parts)
    match = re.search(r'\d{2}[./-]\d{2}[./-]\d{4}', text)
    if match:
        return match.group().replace('-', '.').replace('/', '.')
    match = re.search(r'\d{2}[./-]\d{2}[./-]\d{4}', filename)
    if match:
        return match.group().replace('-', '.').replace('/', '.')
    return None

def replace_negatives(value):
    """Convert parenthesized numbers to negative"""
    if pd.isna(value) or str(value).strip() in ['', '-', '0']:
        return '0'
    val = str(value).strip()
    if '(' in val and ')' in val:
        num = val.replace('(', '').replace(')', '').replace(',', '').strip()
        return f"-{num}"
    return val

# Define financial categories
income_statement_categories = [
    'Të hyrat nga interesi', 'Shpenzimet e interesit', 'Neto të hyrat nga interesi',
    'Të hyrat nga tarifat dhe komisionet', 'Shpenzimet e tarifave dhe komisioneve',
    'Neto të hyrat nga tarifat dhe komisionet', 'Neto të hyrat nga tregtimi me valuta të huaja',
    'Neto të hyrat nga tregtimi', 'Neto të hyrat nga instrumentet tjera financiare',
    'Neto të hyrat (shpenzimet) tjera operative', 'Gjithsej të hyrat',
    'Provizionet për humbjet nga kreditë','Provizione të tjera', 'Fitimi (humbja) para tatimit',
    'Shpenzimet e tatimit në fitim', 'Fitimi (humbja) neto', 'Të ardhurat tjera gjithëpërfshirëse'
]

balance_sheet_categories = [
    "Paraja e gatshme dhe gjendja me BQK-në", "Kërkesat ndaj bankave", "Bonot e thesarit",
    "Mjetet jo qarkulluese të mbajtura për shitje", "Investimet në letra me vlerë",
    "Kreditë dhe paradhëniet ndaj klientëve", "Patundshmëritë dhe pajisjet",
    "Pasuritë e paprekshme", "Pasuritë tatimore të shtyra", "Pasuritë tjera",
    "Gjithsej pasuritë", "Depozitat e klientëve", "Detyrimet ndaj bankave"
]

target_categories = {
    "Income Statement": income_statement_categories,
    "Balance Sheet": balance_sheet_categories,
}

def process_pdf(pdf_path, filename):
    """Process a single PDF and extract financial data"""
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                text = page.extract_text()
                if not text:
                    continue
                dt_report = extract_date(text, pdf_path) or datetime.now().strftime('%Y-%m-%d')
                
                for doc_type, categories in target_categories.items():
                    categories_processed = set()
                    for line in text.split('\n'):
                        for category in categories:
                            if category.lower() in line.lower() and category not in categories_processed:
                                categories_processed.add(category)
                                start_index = line.find(category) + len(category)
                                line_rest = line[start_index:]
                                values = re.findall(r'\(?\d+[,.]?\d*\)?', line_rest)
                                if values and len(values) >= 2:
                                    extracted_data[doc_type].append({
                                        'Category': category,
                                        'PREVIOUS_QUARTER': values[0],
                                        'CURRENT_QUARTER': values[1],
                                        'DT_REPORT': dt_report,
                                        'FILE_NAME': filename
                                    })
        print(f"    Processed: {filename}")
    except Exception as e:
        print(f"    Error processing {filename}: {e}")

def clean_numeric_values(df, prev_col, curr_col):
    for col in [prev_col, curr_col]:
        df[col] = df[col].astype(str).str.replace('-', '0', regex=False)
        df[col] = df[col].apply(replace_negatives)
        df[col] = df[col].str.replace(',', '')
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    return df

# ---------------------------
# MAIN EXECUTION
# ---------------------------

def main():
    global extracted_data
    extracted_data.clear()
    
    # Collect PDF files from both folders
    pdf_files = []
    for category in ['balance-sheet', 'income-statement']:
        category_dir = os.path.join(CURRENT_DIR, category)
        if os.path.exists(category_dir):
            pdf_files.extend([os.path.join(category_dir, f) for f in os.listdir(category_dir)
                              if f.lower().endswith('.pdf')])
    
    print(f" Found {len(pdf_files)} PDF files for processing")
    
    # Process PDFs
    for pdf_path in pdf_files:
        process_pdf(pdf_path, os.path.basename(pdf_path))
    
    # Build DataFrames
    full_bs = pd.DataFrame(extracted_data['Balance Sheet'])
    full_is = pd.DataFrame(extracted_data['Income Statement'])
    
    if not full_bs.empty:
        full_bs.columns = ['BALANCE_CATEGORY', 'PREVIOUS_BALANCE_VALUE', 'CURRENT_BALANCE_VALUE', 'DT_REPORT', 'FILE_NAME']
        full_bs = clean_numeric_values(full_bs, 'PREVIOUS_BALANCE_VALUE', 'CURRENT_BALANCE_VALUE')
    
    if not full_is.empty:
        full_is.columns = ['INCOME_CATEGORY', 'PREVIOUS_INCOME_VALUE', 'CURRENT_INCOME_VALUE', 'DT_REPORT', 'FILE_NAME']
        full_is = clean_numeric_values(full_is, 'PREVIOUS_INCOME_VALUE', 'CURRENT_INCOME_VALUE')
    
    # Save outputs
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    if not full_is.empty:
        full_is.to_csv(os.path.join(OUTPUT_DIR, f"nlb_income_statement_{timestamp}.csv"), index=False)
    if not full_bs.empty:
        full_bs.to_csv(os.path.join(OUTPUT_DIR, f"nlb_balance_sheet_{timestamp}.csv"), index=False)
    
    # Create unified Excel
    if not full_is.empty or not full_bs.empty:
        excel_path = os.path.join(OUTPUT_DIR, f"nlb_financial_report_{timestamp}.xlsx")
        with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
            if not full_is.empty:
                full_is.to_excel(writer, sheet_name='Income_Statement', index=False)
            if not full_bs.empty:
                full_bs.to_excel(writer, sheet_name='Balance_Sheet', index=False)
        print(f" Excel report saved to: {excel_path}")
    
    print("\n Data extraction completed")
    print(f"Income Statement rows: {len(full_is)}")
    print(f"Balance Sheet rows: {len(full_bs)}")
    
    elapsed_time = time.time() - start
    print(f"\n Completed in {elapsed_time:.2f} seconds")
    
    return full_is, full_bs

# ---------------------------
# RUN
# ---------------------------

print(" Starting PDF financial data extraction...")
income_df, balance_df = main()

In [5]:
final_df

""


In [6]:
income_df

,INCOME_CATEGORY,PREVIOUS_INCOME_VALUE,CURRENT_INCOME_VALUE,DT_REPORT,FILE_NAME
0,Të hyrat nga interesi,26352,6314,31.03.2015,Pasqyrat Financiare 31.03.2015.pdf
1,Shpenzimet e interesit,-9603,-1334,31.03.2015,Pasqyrat Financiare 31.03.2015.pdf
2,Neto të hyrat nga interesi,16749,4980,31.03.2015,Pasqyrat Financiare 31.03.2015.pdf
3,Të hyrat nga tarifat dhe komisionet,5620,1322,31.03.2015,Pasqyrat Financiare 31.03.2015.pdf
4,Shpenzimet e tarifave dhe komisioneve,-1157,-297,31.03.2015,Pasqyrat Financiare 31.03.2015.pdf
5,Neto të hyrat nga tarifat dhe komisionet,4463,1025,31.03.2015,Pasqyrat Financiare 31.03.2015.pdf
6,Neto të hyrat nga tregtimi,0,0,31.03.2015,Pasqyrat Financiare 31.03.2015.pdf
7,Neto të hyrat nga instrumentet tjera financiare,446,211,31.03.2015,Pasqyrat Financiare 31.03.2015.pdf
8,Neto të hyrat (shpenzimet) tjera operative,-11218,-3177,31.03.2015,Pasqyrat Financiare 31.03.2015.pdf
9,Gjithsej të hyrat,10441,3039,31.03.2015,Pasqyrat Financiare 31.03.2015.pdf


In [10]:
balance_df

,BALANCE_CATEGORY,PREVIOUS_BALANCE_VALUE,CURRENT_BALANCE_VALUE,DT_REPORT,FILE_NAME
0,Paraja e gatshme dhe gjendja me BQK-në,54465,49353,31.03.2015,Pasqyrat Financiare 31.03.2015.pdf
1,Kërkesat ndaj bankave,93639,91210,31.03.2015,Pasqyrat Financiare 31.03.2015.pdf
2,Bonot e thesarit,0,0,31.03.2015,Pasqyrat Financiare 31.03.2015.pdf
3,Investimet në letra me vlerë,49234,50657,31.03.2015,Pasqyrat Financiare 31.03.2015.pdf
4,Kreditë dhe paradhëniet ndaj klientëve,252161,258047,31.03.2015,Pasqyrat Financiare 31.03.2015.pdf
...,...,...,...,...,...
631,Pasuritë tatimore të shtyra,520,610,31.03.2024,Pasqyrat financiare 31.03.2024.pdf
632,Pasuritë tjera,3933,2212,31.03.2024,Pasqyrat financiare 31.03.2024.pdf
633,Gjithsej pasuritë,1229,757,31.03.2024,Pasqyrat financiare 31.03.2024.pdf
634,Depozitat e klientëve,1008,264,31.03.2024,Pasqyrat financiare 31.03.2024.pdf


📁 Created directory: nlb_financial_data/current/balance-sheet
📁 Created directory: nlb_financial_data/current/income-statement
📁 Created directory: nlb_financial_data
📁 Created directory: nlb_financial_data/original
📁 Created directory: nlb_financial_data/output
🔗 Target URLs: 4 pages
🚀 Starting NLB Bank financial data extraction...

🏦 NLB BANK FINANCIAL DATA EXTRACTION

🌐 DOWNLOAD MODE: Downloading and processing new files
⚠️ Force mode enabled: all files will be downloaded
🔍 Scraping: https://nlb-kos.com/sq/pasqyrat-financiare?page=1
🔍 Scraping: https://nlb-kos.com/sq/pasqyrat-financiare?page=2
🔍 Scraping: https://nlb-kos.com/sq/pasqyrat-financiare?page=3
🔍 Scraping: https://nlb-kos.com/sq/pasqyrat-financiare?page=4
❌ No PDF links found
❌ No data in final DataFrame


In [12]:
# ---------------------------
# CREATE FINAL UNIFIED DATAFRAME
# ---------------------------

unified_dfs = []

# Income → unified format
if not income_df.empty:
    is_unified = income_df.rename(columns={
        'INCOME_CATEGORY': 'ORIGINAL_CATEGORY',
        'PREVIOUS_INCOME_VALUE': 'PREVIOUS_VALUE',
        'CURRENT_INCOME_VALUE': 'CURRENT_VALUE'
    })
    is_unified['CATEGORY_TYPE'] = 'INCOME'
    is_unified['STATEMENT_TYPE'] = 'INCOME_STATEMENT'
    unified_dfs.append(is_unified)

# Balance → unified format
if not balance_df.empty:
    bs_unified = balance_df.rename(columns={
        'BALANCE_CATEGORY': 'ORIGINAL_CATEGORY',
        'PREVIOUS_BALANCE_VALUE': 'PREVIOUS_VALUE',
        'CURRENT_BALANCE_VALUE': 'CURRENT_VALUE'
    })
    bs_unified['CATEGORY_TYPE'] = 'BALANCE'
    bs_unified['STATEMENT_TYPE'] = 'BALANCE_SHEET'
    unified_dfs.append(bs_unified)

# Combine both
if unified_dfs:
    final_df = pd.concat(unified_dfs, ignore_index=True)
    
    # Optional useful fields (same style as your previous script)
    final_df['REPORT_DATE'] = pd.to_datetime(final_df['DT_REPORT'], errors='coerce').dt.strftime('%Y-%m-%d')
    final_df['EXTRACTION_DATE'] = datetime.now().strftime('%Y-%m-%d')
    final_df['DATA_SOURCE'] = 'NLB Bank'
    
    # Sort (optional but recommended)
    final_df = final_df.sort_values(['DT_REPORT', 'STATEMENT_TYPE'])

else:
    final_df = pd.DataFrame()

/var/folders/gg/tbq9nzjd05x7705vgx9mgwch0000gn/T/ipykernel_28179/3015393857.py:34: UserWarning: Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  final_df['REPORT_DATE'] = pd.to_datetime(final_df['DT_REPORT'], errors='coerce').dt.strftime('%Y-%m-%d')


In [13]:
final_df

,ORIGINAL_CATEGORY,PREVIOUS_VALUE,CURRENT_VALUE,DT_REPORT,FILE_NAME,CATEGORY_TYPE,STATEMENT_TYPE,REPORT_DATE,EXTRACTION_DATE,DATA_SOURCE
1229,Paraja e gatshme dhe gjendja me BQK-në,194448,202370,2026-03-18,Pasqyrat Financiare 31 Dhjetor 2025.pdf,BALANCE,BALANCE_SHEET,NaN,2026-03-18,NLB Bank
1230,Kërkesat ndaj bankave,69259,71418,2026-03-18,Pasqyrat Financiare 31 Dhjetor 2025.pdf,BALANCE,BALANCE_SHEET,NaN,2026-03-18,NLB Bank
1231,Bonot e thesarit,0,0,2026-03-18,Pasqyrat Financiare 31 Dhjetor 2025.pdf,BALANCE,BALANCE_SHEET,NaN,2026-03-18,NLB Bank
1232,Investimet në letra me vlerë,170718,170885,2026-03-18,Pasqyrat Financiare 31 Dhjetor 2025.pdf,BALANCE,BALANCE_SHEET,NaN,2026-03-18,NLB Bank
1233,Kreditë dhe paradhëniet ndaj klientëve,1130,91,2026-03-18,Pasqyrat Financiare 31 Dhjetor 2025.pdf,BALANCE,BALANCE_SHEET,NaN,2026-03-18,NLB Bank
...,...,...,...,...,...,...,...,...,...,...
281,Neto të hyrat (shpenzimet) tjera operative,-15078,-20274,31.12.2024,Pasqyrat financiare me 31.12.2024.pdf,INCOME,INCOME_STATEMENT,2024-12-31,2026-03-18,NLB Bank
282,Gjithsej të hyrat,31553,42044,31.12.2024,Pasqyrat financiare me 31.12.2024.pdf,INCOME,INCOME_STATEMENT,2024-12-31,2026-03-18,NLB Bank
283,Provizionet për humbjet nga kreditë,272,-1093,31.12.2024,Pasqyrat financiare me 31.12.2024.pdf,INCOME,INCOME_STATEMENT,2024-12-31,2026-03-18,NLB Bank
284,Shpenzimet e tatimit në fitim,-3138,-3923,31.12.2024,Pasqyrat financiare me 31.12.2024.pdf,INCOME,INCOME_STATEMENT,2024-12-31,2026-03-18,NLB Bank


In [15]:
final_df["REPORT_DATE"] = final_df["REPORT_DATE"].fillna(2026-03-18)

,ORIGINAL_CATEGORY,PREVIOUS_VALUE,CURRENT_VALUE,DT_REPORT,FILE_NAME,CATEGORY_TYPE,STATEMENT_TYPE,REPORT_DATE,EXTRACTION_DATE,DATA_SOURCE
1229,Paraja e gatshme dhe gjendja me BQK-në,194448,202370,2026-03-18,Pasqyrat Financiare 31 Dhjetor 2025.pdf,BALANCE,BALANCE_SHEET,NaN,2026-03-18,NLB Bank
1230,Kërkesat ndaj bankave,69259,71418,2026-03-18,Pasqyrat Financiare 31 Dhjetor 2025.pdf,BALANCE,BALANCE_SHEET,NaN,2026-03-18,NLB Bank
1231,Bonot e thesarit,0,0,2026-03-18,Pasqyrat Financiare 31 Dhjetor 2025.pdf,BALANCE,BALANCE_SHEET,NaN,2026-03-18,NLB Bank
1232,Investimet në letra me vlerë,170718,170885,2026-03-18,Pasqyrat Financiare 31 Dhjetor 2025.pdf,BALANCE,BALANCE_SHEET,NaN,2026-03-18,NLB Bank
1233,Kreditë dhe paradhëniet ndaj klientëve,1130,91,2026-03-18,Pasqyrat Financiare 31 Dhjetor 2025.pdf,BALANCE,BALANCE_SHEET,NaN,2026-03-18,NLB Bank
1234,Patundshmëritë dhe pajisjet,11451,12447,2026-03-18,Pasqyrat Financiare 31 Dhjetor 2025.pdf,BALANCE,BALANCE_SHEET,NaN,2026-03-18,NLB Bank
1235,Pasuritë e paprekshme,1520,1856,2026-03-18,Pasqyrat Financiare 31 Dhjetor 2025.pdf,BALANCE,BALANCE_SHEET,NaN,2026-03-18,NLB Bank
1236,Pasuritë tatimore të shtyra,614,729,2026-03-18,Pasqyrat Financiare 31 Dhjetor 2025.pdf,BALANCE,BALANCE_SHEET,NaN,2026-03-18,NLB Bank
1237,Pasuritë tjera,4007,6132,2026-03-18,Pasqyrat Financiare 31 Dhjetor 2025.pdf,BALANCE,BALANCE_SHEET,NaN,2026-03-18,NLB Bank
1238,Gjithsej pasuritë,1582,108,2026-03-18,Pasqyrat Financiare 31 Dhjetor 2025.pdf,BALANCE,BALANCE_SHEET,NaN,2026-03-18,NLB Bank


In [16]:
final_df["REPORT_DATE"] = final_df["REPORT_DATE"].fillna('2026-03-18')

In [17]:
final_df.to_csv("NLB.csv")